In [ ]:
]activate ../../../

In [ ]:
using Revise
includet("./base.jl")

In [ ]:
includet("../../../scripts/figures_util.jl")

using GLMakie
using CairoMakie
GLMakie.activate!()

# Loading all the runs into one dataframe

In [ ]:
maindir = "./main1"

files = filter(readdir(maindir; join=true)) do f
    endswith(f, ".jld2") && !endswith(f, "_fit.jld2")
end
sort!(files; by=f -> parse(Int, match(r"gi(\d+)", basename(f))[1]))

adf = mapreduce(vcat, files) do fname
    metadata, df = load(fname, "metadata", "df")
    insertcols!(df, 1,
        :file => basename(fname),
        :row_id => 1:nrow(df),
        :K => metadata.K,
        :l => metadata.l,
        :p => metadata.p,
    )
end

@show nrow(adf) length(files)
adf

## Checks

In [ ]:
# what came out, per (K, l, p) group
combine(groupby(adf, [:K, :l, :p]), nrow => :num_systems)

In [ ]:
# how many systems are usable
@show count(adf.retcodes .== ReturnCode.Success)
@show count(adf.maxresids .< 1e-10)
@show count(adf.num_surv .!= 0)
combine(groupby(adf, [:K, :l, :p]),
    :num_surv => (x -> count(!=(0), x)) => :num_surviving,
)